# Modul 3 · SQL und Datenqualität
**Data Science 1** · linkit courses

Die Nordwerk GmbH schickt dir den ungeprüften Abzug ihrer Werkssysteme als SQLite-Datenbank. Bevor jemand damit rechnet, sollst du herausfinden, wie viel man diesen Daten glauben kann, und sie so bereinigen, dass die Bereinigung beim nächsten Abzug wieder läuft.

**Lernziele**
1. Abfragen mit SELECT, WHERE, GROUP BY, JOIN und Fensterfunktionen schreiben
2. Mechanismen fehlender Werte unterscheiden und ihre Folgen abschätzen
3. Ausreißer robust erkennen
4. Datenprüfungen als Code formulieren
5. personenbezogene Daten pseudonymisieren und sparsam halten

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis fünf Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 5, TODO 7 und 8 sowie TODO 10 bis 12. TODO 6, TODO 9 und die Bonusaufgaben vertiefen.
- SQL-Abfragen schreibst du als Text in dreifachen Anführungszeichen und führst sie mit `sql("""…""")` aus. Das Ergebnis ist ein DataFrame.
- Die Rohdatenbank öffnest du nur zum Lesen. Beispiele, die etwas schreiben, laufen in einer kleinen Datenbank im Arbeitsspeicher.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["nordwerk_roh.db", "produktion.csv", "stoerungen.csv", "maschinen.csv", "linien.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Relationale Daten: Tabellen, Schlüssel, Beziehungen
*Lerneinheit 1 · Skript Abschnitt 1*

Eine relationale Datenbank speichert Daten in Tabellen. Ein **Primärschlüssel** bestimmt jede Zeile seiner Tabelle eindeutig, ein **Fremdschlüssel** verweist auf den Primärschlüssel einer anderen Tabelle. Eine Maschine hat viele Störungsmeldungen, jede Meldung gehört zu einer Maschine: eine **1:n-Beziehung**. Name und Linie einer Maschine stehen deshalb einmal in der Maschinentabelle und nicht in jeder Meldung. So kann keine Meldung einer Maschine eine andere Linie zuschreiben als die übrigen.

Die nächste Zelle legt die kleinen Tabellen `stamm` und `meldungen` aus dem Skript an, schreibt sie in eine Datenbank `klein` im Arbeitsspeicher und öffnet die Rohdatenbank als `con`, nur zum Lesen. Die Funktion `sql` führt eine Abfrage aus, ohne zweites Argument an der Rohdatenbank.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd

stamm = pd.DataFrame({
    "maschine": ["M-03", "M-05", "M-07", "M-12"],
    "name": ["CNC-Drehmaschine", "Roboter-Schweißzelle", "Prüfstand", "Entgratstation"],
    "linie": [1, 3, 4, 2],
})
meldungen = pd.DataFrame({
    "id": ["S-1", "S-2", "S-3", "S-4", "S-5", "S-6", "S-7"],
    "datum": ["2025-03-03", "2025-03-03", "2025-03-03", "2025-03-04", "2025-03-04", "2025-03-05", "2025-03-05"],
    "schicht": ["Früh", "Spät", "Nacht", "Früh", "Nacht", "Früh", "Nacht"],
    "maschine": ["M-03", "M-05", "M-03", "M-07", "M3", "M-03", "M-05"],
    "kategorie": ["mechanisch", "elektrisch", None, "Software", "mechanisch", "mechanisch", None],
    "stillstand_min": [40, 25, 60, 15, 35, 50, 20],
})

klein = sqlite3.connect(":memory:")            # Beispieldatenbank im Arbeitsspeicher, hier darf geschrieben werden
klein.execute("CREATE TABLE maschinen (maschine TEXT PRIMARY KEY, name TEXT, linie INTEGER)")
klein.execute("""CREATE TABLE meldungen (id TEXT PRIMARY KEY, datum TEXT, schicht TEXT,
    maschine TEXT REFERENCES maschinen (maschine), kategorie TEXT, stillstand_min INTEGER)""")
stamm.to_sql("maschinen", klein, if_exists="append", index=False)
meldungen.to_sql("meldungen", klein, if_exists="append", index=False)

con = sqlite3.connect("file:daten/nordwerk_roh.db?mode=ro", uri=True)    # Rohdatenbank, nur lesen

def sql(abfrage, db=None):
    """Führt eine SQL-Abfrage aus und liefert das Ergebnis als DataFrame (ohne db: die Rohdatenbank)."""
    return pd.read_sql_query(abfrage, con if db is None else db)

print(sql("SELECT * FROM meldungen", klein))

Die Datenbank prüft den Primärschlüssel bei jedem Einfügen. Fremdschlüssel prüft SQLite nur, wenn jemand das mit `PRAGMA foreign_keys = ON` einschaltet. Deshalb steht die Meldung S-5 mit der unbekannten Maschine `M3` in der Tabelle. `PRAGMA foreign_key_check` findet sie nachträglich.

In [ ]:
try:
    with klein:                                 # verwirft die Änderung, wenn ein Fehler auftritt
        klein.execute("INSERT INTO maschinen VALUES ('M-03', 'Doppelt', 2)")
except sqlite3.IntegrityError as fehler:
    print("IntegrityError:", fehler)
print(sql("PRAGMA foreign_key_check", klein))

Jede SQLite-Datenbank führt in der Tabelle `sqlite_master` Buch über ihren Inhalt. `PRAGMA table_info(tabelle)` zeigt die Spalten einer Tabelle mit Typ und der Spalte `pk`, die für Spalten des Primärschlüssels 1 ist.

In [ ]:
print(sql("SELECT type, name FROM sqlite_master"))
print(sql("PRAGMA table_info(stoerungen)")[["name", "type", "pk"]])

### TODO 1 · Die Datenbank erkunden
Erstelle `uebersicht`: einen DataFrame mit einer Zeile je Tabelle (nur Einträge mit `type = 'table'` in `sqlite_master`, in deren Reihenfolge) und den Spalten
- `tabelle`: Name der Tabelle,
- `zeilen`: Anzahl der Zeilen, mit `SELECT COUNT(*) …`,
- `spalten`: Anzahl der Spalten, als Länge des Ergebnisses von `PRAGMA table_info(…)`.

Den Tabellennamen setzt du mit einem f-String in die Abfrage ein, etwa `sql(f"SELECT COUNT(*) AS n FROM {name}")`. Eine Liste von Dictionaries wird mit `pd.DataFrame(liste)` zur Tabelle.

In [ ]:
# TODO 1: eine Zeile je Tabelle mit Name, Zahl der Zeilen und Zahl der Spalten
uebersicht = None

print(uebersicht)

In [ ]:
check("TODO 1a sieben Tabellen", lambda: len(uebersicht) == 7 and list(uebersicht.columns) == ["tabelle", "zeilen", "spalten"])
check("TODO 1b Zeilen", lambda: uebersicht.set_index("tabelle")["zeilen"].to_dict() == {
    "linien": 4, "maschinen": 20, "schichten": 5816, "stoerungen": 14434, "wartungen": 1042, "auftraege": 2339, "bedienende": 126})
check("TODO 1c Spalten", lambda: uebersicht["spalten"].sum() == 52 and uebersicht.set_index("tabelle").loc["schichten", "spalten"] == 9)

Das Datenmodell der Rohdatenbank. Fremdschlüssel sind in der Datenbank nicht hinterlegt, und für `schichten` und `stoerungen` kennt sie auch keinen Primärschlüssel. Niemand hat also verhindert, dass Zeilen doppelt vorkommen oder ins Leere verweisen.

| Tabelle | Schlüssel | verweist auf |
|---|---|---|
| `linien` | `linie` | |
| `maschinen` | `maschine` | `linien` über `linie` |
| `schichten` | `datum`, `linie`, `schicht` (nicht hinterlegt) | `linien` über `linie` |
| `stoerungen` | `id` (nicht hinterlegt) | `maschinen` über `maschine` |
| `wartungen` | `wartung_id` | `maschinen` über `maschine` |
| `auftraege` | `auftrag_id` | `linien` über `linie` |
| `bedienende` | `personal_nr` | `linien` über `linie` |

`produktion.csv` aus Modul 2 hat 5.788 Zeilen, `schichten` hat 5.816. Woher die 28 zusätzlichen Zeilen kommen, findest du in Abschnitt 3 heraus.

## 2 · SQL-Grundlagen: SELECT, WHERE, GROUP BY, HAVING
*Lerneinheit 2 · Skript Abschnitt 2*

```sql
SELECT spalte, COUNT(*) AS anzahl     -- Spalten und Aggregate, AS vergibt Namen
FROM tabelle
WHERE bedingung                       -- filtert Zeilen vor dem Gruppieren
GROUP BY spalte
HAVING COUNT(*) >= 10                 -- filtert Gruppen nach dem Gruppieren
ORDER BY anzahl DESC                  -- DESC: absteigend
LIMIT 5
```

Die Datenbank wertet logisch in der Reihenfolge `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `SELECT`, `ORDER BY`, `LIMIT` aus. Deshalb darf in `WHERE` keine Aggregatfunktion stehen, und Namen aus `SELECT` sind dort im SQL-Standard noch unbekannt. Text steht in einfachen Anführungszeichen, `DISTINCT` entfernt doppelte Ergebniszeilen.

In [ ]:
# Die Beispiele aus dem Skript, Abschnitte 2.1 und 2.2
print(sql("""
SELECT id, maschine, stillstand_min
FROM meldungen
WHERE stillstand_min >= 30
ORDER BY stillstand_min DESC
LIMIT 3
""", klein))
print(sql("""
SELECT maschine, COUNT(*) AS meldungen, SUM(stillstand_min) AS stillstand
FROM meldungen
GROUP BY maschine
HAVING COUNT(*) >= 2
ORDER BY stillstand DESC
""", klein))

Wie viele Zeilen hat `schichten` je Schicht? `quote` setzt jeden Text in Anführungszeichen, so werden Leerzeichen am Rand sichtbar.

In [ ]:
print(sql("""
SELECT quote(schicht) AS schicht, COUNT(*) AS zeilen
FROM schichten
GROUP BY schicht
ORDER BY zeilen DESC
"""))

Elf Schreibweisen statt drei Schichten. Für SQL sind `'Spät'`, `'spät'` und `'Spät '` verschiedene Werte. Vereinheitlichen wirst du sie in Abschnitt 6.

### TODO 2 · Störungen je Maschine mit GROUP BY und HAVING
Schreibe eine Abfrage an die Tabelle `stoerungen` und speichere ihr Ergebnis als `je_maschine`. Gesucht sind je `maschine`
- `meldungen`: die Anzahl der Meldungen,
- `stunden`: der gesamte Stillstand in Stunden, mit `ROUND(…, 1)` auf eine Nachkommastelle gerundet,

nur für Maschinen mit mindestens 100 Meldungen, absteigend sortiert nach `meldungen`.

In [ ]:
# TODO 2
je_maschine = None

print(je_maschine)

In [ ]:
check("TODO 2a 20 Maschinen, absteigend", lambda: len(je_maschine) == 20 and je_maschine["meldungen"].is_monotonic_decreasing)
check("TODO 2b M-03 vorn", lambda: je_maschine.iloc[0]["maschine"] == "M-03" and je_maschine.iloc[0]["meldungen"] == 1285
      and math.isclose(je_maschine.iloc[0]["stunden"], 911.5))
check("TODO 2c Summe der Meldungen", lambda: je_maschine["meldungen"].sum() == 14278)

In [ ]:
if je_maschine is None:
    print("⏳ Erst TODO 2 lösen.")
else:
    alle = sql("SELECT COUNT(*) AS n FROM stoerungen")["n"].iloc[0]
    print(f"{alle} Meldungen in der Tabelle, {je_maschine['meldungen'].sum()} in je_maschine, es fehlen {alle - je_maschine['meldungen'].sum()}.")

`HAVING` hat Gruppen mit weniger als 100 Meldungen ausgeblendet. Welche Kennungen das sind, zeigt Abschnitt 3.

**NULL** ist der fehlende Wert in SQL und bedeutet „unbekannt“. Ein Vergleich mit `NULL` ergibt wieder „unbekannt“, und `WHERE` behält nur Zeilen, deren Bedingung wahr ist. Deshalb liefert `WHERE kategorie = NULL` nie eine Zeile, richtig ist `WHERE kategorie IS NULL`. `COUNT(*)` zählt Zeilen, `COUNT(spalte)` nur Zeilen, in denen die Spalte nicht `NULL` ist. Eine zweite Falle: Teilt SQLite zwei ganze Zahlen, ist das Ergebnis eine ganze Zahl.

In [ ]:
print(sql("""
SELECT COUNT(*) AS alle, COUNT(kategorie) AS mit_kategorie,
       SUM(kategorie = 'mechanisch') AS mechanisch,
       SUM(kategorie != 'mechanisch') AS nicht_mechanisch,
       SUM(kategorie IS NULL) AS ohne
FROM meldungen
""", klein))
print(sql("SELECT COUNT(*) AS n FROM stoerungen WHERE kategorie = NULL"))
print(sql("""
SELECT linie, SUM(ausschuss_stueck) / SUM(ist_stueck) AS quote_falsch,
       ROUND(1.0 * SUM(ausschuss_stueck) / SUM(ist_stueck), 4) AS quote
FROM schichten
GROUP BY linie
"""))

### TODO 3 · COUNT(*) gegen COUNT(kategorie)
Passieren nachts mehr Bedienfehler als früh oder spät? Bei manchen Meldungen fehlt die Kategorie, also schau zuerst, wo. Speichere als `fehlend_je_schicht` eine Abfrage mit einer Zeile je `schicht` und den Spalten
- `meldungen`: alle Meldungen,
- `mit_kategorie`: Meldungen mit Kategorie,
- `ohne_kategorie`: Meldungen ohne Kategorie,
- `anteil_ohne`: `ohne_kategorie` geteilt durch `meldungen` als Zahl zwischen 0 und 1. Achte auf die ganzzahlige Division.
- `bedienung`: die Zahl der Meldungen mit der Kategorie `'Bedienung'`. Ein Vergleich liefert in SQLite 1 oder 0, `SUM(bedingung)` zählt also die wahren Fälle.

In [ ]:
# TODO 3
fehlend_je_schicht = None

print(fehlend_je_schicht)

In [ ]:
check("TODO 3a drei Schichten", lambda: len(fehlend_je_schicht) == 3 and fehlend_je_schicht["meldungen"].sum() == 14434)
check("TODO 3b ohne Kategorie", lambda: fehlend_je_schicht.set_index("schicht")["ohne_kategorie"].to_dict() == {"Früh": 82, "Nacht": 1358, "Spät": 84})
check("TODO 3c Anteil nachts", lambda: math.isclose(fehlend_je_schicht.set_index("schicht").loc["Nacht", "anteil_ohne"], 0.275233, abs_tol=1e-6))
check("TODO 3d Bedienfehler gezählt", lambda: fehlend_je_schicht.set_index("schicht")["bedienung"].to_dict() == {"Früh": 697, "Nacht": 629, "Spät": 701})

Nachts fehlt die Kategorie in mehr als jeder vierten Meldung, früh und spät in weniger als jeder fünfzigsten. Und nach den vollständigen Zeilen passieren nachts die wenigsten Bedienfehler. Ob das stimmt, klärt Abschnitt 4.

## 3 · JOINs und Fensterfunktionen
*Lerneinheit 3 · Skript Abschnitt 3*

- `JOIN` (kurz für `INNER JOIN`) verknüpft zwei Tabellen über eine Bedingung `ON a.schluessel = b.schluessel` und behält nur Zeilen mit Partner, wie `how="inner"` bei `merge`.
- `LEFT JOIN` behält alle Zeilen der linken Tabelle und füllt fehlende Partner mit `NULL`, wie `how="left"`.
- **Anti-Join:** `LEFT JOIN … WHERE rechts.schluessel IS NULL` findet die Zeilen links, die keinen Partner haben.
- Kurze Aliasnamen wie `FROM stoerungen AS s` sparen Schreibarbeit.

In [ ]:
print(sql("""
SELECT m.id, m.maschine, s.name, s.linie
FROM meldungen AS m
LEFT JOIN maschinen AS s ON m.maschine = s.maschine
""", klein))
print(sql("""
SELECT (SELECT COUNT(*) FROM stoerungen) AS meldungen,
       (SELECT COUNT(*) FROM stoerungen AS s JOIN maschinen AS m ON s.maschine = m.maschine) AS inner_join,
       (SELECT COUNT(*) FROM stoerungen AS s LEFT JOIN maschinen AS m ON s.maschine = m.maschine) AS left_join
"""))

Der `INNER JOIN` verliert Meldungen, ohne etwas zu melden. Der Anti-Join in der kleinen Datenbank:

In [ ]:
print(sql("""
SELECT m.id, m.maschine
FROM meldungen AS m
LEFT JOIN maschinen AS s ON m.maschine = s.maschine
WHERE s.maschine IS NULL
""", klein))

stoerungen_roh = sql("SELECT * FROM stoerungen")      # die Rohdaten als DataFrame, wie im Skript
maschinen = sql("SELECT * FROM maschinen")
print(stoerungen_roh.shape, maschinen.shape)

### TODO 4 · Unbekannte Maschinen finden und Kennungen normieren
1. `unbekannt`: Schreibe einen Anti-Join an die Rohdatenbank. Gesucht ist je Kennung `maschine` aus `stoerungen`, die es in `maschinen` nicht gibt, die Zahl der Meldungen als Spalte `meldungen`.
2. Schau dir die Kennungen an und schreibe die Funktion `normiere_maschine(spalte)` (die Werkzeuge zeigt das Skript in Abschnitt 3.2 an der Beispieltabelle), die eine Series von Kennungen wie `M3`, `M03` oder `m-03` auf die Form `M-03` bringt. Nützlich sind `str.strip()`, `str.upper()`, `str.replace("-", "")`, `str[1:]` und `str.zfill(2)`.
3. `ohne_partner`: die Zahl der Meldungen in `stoerungen_roh`, deren normierte Kennung nicht in `maschinen["maschine"]` vorkommt. `~serie.isin(liste)` ist der Anti-Join in pandas.

In [ ]:
# TODO 4a: Anti-Join in SQL
unbekannt = None

# TODO 4b: Kennungen auf die Form M-NN bringen
def normiere_maschine(spalte):
    return spalte          # ersetze diese Zeile

# TODO 4c
ohne_partner = None

print(unbekannt)
print(ohne_partner)

In [ ]:
check("TODO 4a Anti-Join", lambda: len(unbekannt) == 45 and unbekannt["meldungen"].sum() == 156)
check("TODO 4b normiere_maschine", lambda: normiere_maschine(pd.Series(["M-03", "M3", "M03", "m-03", "M12", "m-20"])).tolist()
      == ["M-03", "M-03", "M-03", "M-03", "M-12", "M-20"])
check("TODO 4c jede Meldung findet ihre Maschine", lambda: ohne_partner == 0)

`RIGHT JOIN` und `FULL OUTER JOIN` kennt SQLite erst seit Version 3.39. Du brauchst beide nicht: Ein `RIGHT JOIN` ist ein `LEFT JOIN` mit vertauschten Tabellen, ein `FULL OUTER JOIN` ein `LEFT JOIN` plus die Zeilen der rechten Tabelle ohne Partner, angehängt mit `UNION ALL`:

In [ ]:
print(sql("""
SELECT m.id, m.maschine AS gemeldet, s.maschine AS stamm
FROM meldungen AS m LEFT JOIN maschinen AS s ON m.maschine = s.maschine
UNION ALL
SELECT NULL, NULL, s.maschine
FROM maschinen AS s LEFT JOIN meldungen AS m ON m.maschine = s.maschine
WHERE m.id IS NULL
""", klein))

**Fensterfunktionen** rechnen je Gruppe wie `GROUP BY`, behalten aber jede Zeile, ähnlich wie `transform` in Modul 2. `OVER (PARTITION BY … ORDER BY …)` beschreibt das Fenster:
- `ROW_NUMBER()` nummeriert die Zeilen jeder Gruppe ab 1,
- `LAG(spalte)` liefert den Wert der vorigen Zeile in der Gruppe, für die erste `NULL`,
- `SUM(spalte) OVER (… ORDER BY …)` bildet eine laufende Summe.

`julianday(datum)` macht aus einem Datumstext eine Tageszahl. Die Differenz zweier solcher Zahlen ist der Abstand in Tagen.

In [ ]:
print(sql("""
SELECT id, maschine, datum, stillstand_min,
       ROW_NUMBER() OVER (PARTITION BY maschine ORDER BY datum, id) AS nr,
       LAG(datum) OVER (PARTITION BY maschine ORDER BY datum, id) AS vorher,
       SUM(stillstand_min) OVER (PARTITION BY maschine ORDER BY datum, id) AS laufend
FROM meldungen
ORDER BY maschine, nr
""", klein))

# Wie im Skript, Abschnitt 3.3: Tage zwischen zwei erledigten Wartungen derselben Maschine
abstaende = sql("""
SELECT maschine, datum, art,
       julianday(datum) - julianday(LAG(datum) OVER (PARTITION BY maschine ORDER BY datum, wartung_id)) AS tage
FROM wartungen
WHERE status = 'erledigt'
""")
print(abstaende.head(4))
print(len(abstaende), "Wartungen, im Mittel", round(abstaende["tage"].mean(), 1), "Tage Abstand")

### TODO 5 · Wird der Wartungsplan eingehalten?
Laut Wartungsplan im Datenblatt der Rohdatenbank (`DATENBLATT_ROH.md`, Tabelle `wartungen`) kommt jede Maschine etwa alle 6 Wochen zur Inspektion, alle 8 Wochen zur Schmierung, alle 13 Wochen zur Kalibrierung und zum Tausch der Verschleißteile und einmal im Jahr (52 Wochen) zur Großwartung.
1. `abstaende_art`: eine Tabelle der erledigten Wartungen mit den Spalten `maschine`, `art`, `datum` und `tage`. `tage` ist der Abstand in Tagen zur vorigen Wartung, und zwar so gemessen, dass du ihn mit dem Plan vergleichen kannst.
2. `mittel_art`: der mittlere Abstand in Tagen je Art, als Series mit der Art als Index.

In [ ]:
# TODO 5
abstaende_art = None
mittel_art = None

print(mittel_art)

In [ ]:
check("TODO 5a abstaende_art", lambda: len(abstaende_art) == 965 and abstaende_art["tage"].isna().sum() == 100)
check("TODO 5b mittel_art", lambda: math.isclose(mittel_art["Inspektion"], 42.53, abs_tol=0.01)
      and math.isclose(mittel_art["Großwartung"], 357.86, abs_tol=0.01))

In [ ]:
if mittel_art is None:
    print("⏳ Erst TODO 5 lösen.")
else:
    plan = pd.Series({"Inspektion": 6, "Schmierung": 8, "Kalibrierung": 13, "Verschleißteile": 13, "Großwartung": 52})
    print(pd.DataFrame({"plan_wochen": plan, "ist_wochen": (mittel_art / 7).round(2)}))

Im Mittel hält die Instandhaltung den Plan sehr genau ein, die einzelnen Abstände weichen um bis zu 18 % davon ab. Mit `MIN` und `MAX` je Art kannst du das selbst nachprüfen.

### TODO 6 · Dubletten zählen und markieren
1. `dubletten_schichten`: Zahl der Zeilen von `schichten` minus Zahl der verschiedenen Zeilen (`SELECT DISTINCT *`), als ganze Zahl aus einer SQL-Abfrage.
2. `markiert`: Nummeriere in `stoerungen` mit `ROW_NUMBER()` inhaltsgleiche Meldungen, gleich in allen Spalten außer `id`, sortiert nach `id`. Das Ergebnis hat die Spalten `id`, `datum`, `maschine`, `stillstand_min` und `nr`.
3. `n_markiert`: die Zahl der Meldungen mit `nr` größer als 1.

In [ ]:
# TODO 6
dubletten_schichten = None
markiert = None
n_markiert = None

print(dubletten_schichten, n_markiert)

In [ ]:
check("TODO 6a exakte Dubletten in schichten", lambda: dubletten_schichten == 28)
check("TODO 6b markiert", lambda: len(markiert) == 14434 and markiert["nr"].max() == 2)
check("TODO 6c n_markiert", lambda: n_markiert == 195)

In [ ]:
if markiert is None:
    print("⏳ Erst TODO 6 lösen.")
else:
    zweite = markiert[markiert["nr"] > 1]
    print("Kennungen der markierten Meldungen:", zweite["id"].min(), "bis", zweite["id"].max())
    print("davon ab S-14264:", (zweite["id"] >= "S-14264").sum())

Die meisten markierten Meldungen haben Kennungen am Ende der Tabelle, aber nicht alle. Ob alle 195 Kopien sind, klärt Abschnitt 6 mit dem Abgleich gegen Modul 2.

## 4 · Fehlende Werte: MCAR, MAR, MNAR und was Löschen anrichtet
*Lerneinheit 4 · Skript Abschnitt 4*

Sei $R = 1$, wenn ein Wert fehlt. Nach Rubin (1976) unterscheidest du drei Mechanismen:
- **MCAR** (missing completely at random): Ob ein Wert fehlt, hängt von nichts in den Daten ab. Löschen unvollständiger Zeilen ist unverzerrt, du verlierst nur Daten.
- **MAR** (missing at random): Es hängt nur von beobachteten Werten ab, etwa von der Schicht. Löschen verzerrt, aber der Anteil innerhalb jeder Schicht bleibt unverzerrt. Mit ihm rechnest du die Anzahl hoch: $\hat N_s = n_s \cdot \hat P(K = k \mid S = s)$, mit $n_s$ allen Meldungen der Schicht $s$.
- **MNAR** (missing not at random): Es hängt vom fehlenden Wert selbst ab. Löschen verzerrt, und die Daten allein bieten keine Abhilfe.

`AVG` in SQL und `mean` in pandas überspringen fehlende Werte. Das ist dasselbe wie Löschen.

In [ ]:
# Die acht Sommerschichten aus dem Skript, Abschnitt 4.1
wahr = pd.Series([23.0, 26.0, 21.0, 24.0, 28.0, 22.0, 25.0, 23.0])
mcar = wahr.copy()
mcar[[2, 6]] = np.nan                 # zwei zufällig gewählte Lücken
mnar = wahr.where(wahr <= 24.5)       # der Logger fällt über 24,5 °C aus
print(pd.DataFrame({"wahr": wahr, "mcar": mcar, "mnar": mnar}))
print("Mittelwerte:", wahr.mean(), round(mcar.mean(), 2), mnar.mean())

Für die wahren Werte brauchst du die geprüften Dateien aus Modul 2. `SCHICHT_TYP` kennst du von dort. Die geprüften Störungen heißen `stoerungen_m2`, damit sie nicht mit der Tabelle `stoerungen` der Rohdatenbank verwechselt werden. Zuerst `personal`, das in 74 Schichten fehlt:

In [ ]:
SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
produktion = pd.read_csv("daten/produktion.csv", parse_dates=["datum"], dtype={"schicht": SCHICHT_TYP})
stoerungen_m2 = pd.read_csv("daten/stoerungen.csv", parse_dates=["datum"], dtype={"schicht": SCHICHT_TYP})

personal_roh = sql("SELECT AVG(personal) AS m FROM (SELECT DISTINCT * FROM schichten)")["m"].iloc[0]
print("Personal je Schicht, Lücken gelöscht:", round(personal_roh, 4), "| wahr:", round(produktion["personal"].mean(), 4))

Unter MCAR ist Löschen unverzerrt. Die Kategorie fehlt dagegen nachts viel häufiger (TODO 3), sie fehlt also abhängig von der Schicht: MAR. Was das für die Frage nach den Bedienfehlern bedeutet, rechnest du in TODO 7 nach. Bei der Hallentemperatur häufen sich die Lücken in einer anderen Weise. Ihr Anteil je Monat (ohne den März 2025, dessen Datum in einem anderen Format steht):

In [ ]:
fehlend_monat = sql("""
SELECT substr(datum, 6, 2) AS monat, AVG(hallentemp_c IS NULL) AS anteil_ohne_temp
FROM (SELECT DISTINCT * FROM schichten)
WHERE datum LIKE '____-__-__'
GROUP BY monat
""")
ax = fehlend_monat.plot.bar(x="monat", y="anteil_ohne_temp", legend=False, figsize=(8, 3),
                            title="Anteil der Schichten ohne Hallentemperatur je Monat")
ax.set_xlabel("Monat")
ax.set_ylabel("Anteil ohne Wert")
print(fehlend_monat.round(3).to_string(index=False))

Im Sommer fehlt fast jeder vierte Wert, im Winter fast keiner. Der Temperaturlogger fällt bei Hitze aus: Die Lücken hängen vom fehlenden Wert selbst ab, also MNAR.

### TODO 7 · Die Verzerrung durch Löschen messen
**a) MAR: Passieren nachts mehr Bedienfehler?**
1. `bedienung_hoch`: die hochgerechnete Zahl der Bedienfehler je Schicht, als Series mit der Schicht als Index. Rechne alle Meldungen der Schicht mal den Anteil der Bedienfehler unter den Meldungen mit Kategorie. Alle Zutaten stehen in `fehlend_je_schicht` aus TODO 3, mit `.set_index("schicht")` wird die Schicht zum Index.
2. `bedienung_wahr`: die wahre Zahl der Bedienfehler je Schicht aus `stoerungen_m2`, ebenfalls mit der Schicht als Index. `.astype(str)` macht aus der Kategorie `schicht` Text, damit die Indizes zusammenpassen.

**b) MNAR: Wie warm war der Sommer?**
3. `sommer_roh`: die mittlere Hallentemperatur aller Schichten im Juli und August beider Jahre aus der Rohdatenbank, ohne exakte Dubletten (`SELECT DISTINCT *` in einer Unterabfrage), Lücken übersprungen. Den Monat liefert `substr(datum, 6, 2)`.
4. `sommer_wahr`: dieselbe Größe aus `produktion` (`.dt.month`).
5. `verzerrung`: `sommer_roh - sommer_wahr`.
6. `fehlend_wahr`: der mittlere wahre Wert der fehlenden Schichten. Weil die beobachteten Werte mit Modul 2 übereinstimmen, gilt mit $n$ Schichten, davon $n_{\text{obs}}$ mit Wert: $\bar x_{\text{fehlend}} = (n \cdot \bar x_{\text{wahr}} - n_{\text{obs}} \cdot \bar x_{\text{roh}})/(n - n_{\text{obs}})$.

In [ ]:
# TODO 7a
bedienung_hoch = None
bedienung_wahr = None

# TODO 7b
sommer_roh = None
sommer_wahr = None
verzerrung = None
fehlend_wahr = None

print(bedienung_hoch)
print(sommer_roh, sommer_wahr, verzerrung, fehlend_wahr)

In [ ]:
check("TODO 7a bedienung_hoch", lambda: math.isclose(bedienung_hoch["Nacht"], 867.9, abs_tol=0.1)
      and math.isclose(bedienung_hoch["Früh"], 709.0, abs_tol=0.1) and bedienung_hoch.idxmax() == "Nacht")
check("TODO 7a bedienung_wahr", lambda: bedienung_wahr.to_dict() == {"Früh": 700, "Nacht": 887, "Spät": 702})
check("TODO 7b sommer_roh", lambda: math.isclose(sommer_roh, 23.1221, abs_tol=1e-4))
check("TODO 7b sommer_wahr", lambda: math.isclose(sommer_wahr, 23.5572, abs_tol=1e-4))
check("TODO 7b verzerrung", lambda: math.isclose(verzerrung, -0.4351, abs_tol=1e-4))
check("TODO 7b fehlend_wahr", lambda: math.isclose(fehlend_wahr, 25.377, abs_tol=1e-3))

In [ ]:
if bedienung_hoch is None or bedienung_wahr is None or "bedienung" not in getattr(fehlend_je_schicht, "columns", []):
    print("⏳ Erst TODO 3 und TODO 7a lösen.")
else:
    print(pd.DataFrame({"gezaehlt": fehlend_je_schicht.set_index("schicht")["bedienung"],
                        "hochgerechnet": bedienung_hoch.round(1), "wahr": bedienung_wahr}))

Gezählt hat die Nacht die wenigsten Bedienfehler, wahr hat sie die meisten: Wer die Frage mit den vollständigen Zeilen beantwortet, kommt zum falschen Schluss. Hochgerechnet mit den Anteilen je Schicht hat die Nacht wieder die meisten. Ganz genau trifft die Hochrechnung nicht, weil der Anteil der Nacht nur auf den Meldungen mit Kategorie beruht und zufällig schwankt, und die 171 doppelten Meldungen aus Abschnitt 6 stecken noch in den Rohdaten.

Unter MNAR hilft kein Hochrechnen. Ohne die Lücken ist der Sommer fast ein halbes Grad zu kühl, denn die fehlenden Schichten waren im Mittel über 25 °C warm. Aus den Rohdaten allein ließe sich das nicht beheben, kein beobachteter Wert verrät, wie heiß es in den fehlenden Schichten war.

Zwei weitere Fallen: Eine Lücke ist nicht immer ein Fehler, und ein Wert ist nicht immer ein Wert.

In [ ]:
print(sql("""
SELECT status, COUNT(*) AS auftraege, COUNT(geliefert_am) AS mit_lieferdatum
FROM auftraege
GROUP BY status
"""))
print(sql("""
SELECT SUM(ausschuss_stueck = -1) AS platzhalter,
       ROUND(AVG(ausschuss_stueck), 4) AS mittel_roh,
       ROUND(AVG(NULLIF(ausschuss_stueck, -1)), 4) AS mittel_ohne
FROM (SELECT DISTINCT * FROM schichten)
"""))

# Mittelwert-Imputation (Skript, Abschnitt 4.5): Die Verzerrung bleibt, die Streuung schrumpft.
gefuellt = mnar.fillna(mnar.mean())
print("Mittel", gefuellt.mean(), "| Standardabweichung beobachtet", round(mnar.std(), 3),
      "gefüllt", round(gefuellt.std(), 3), "wahr", round(wahr.std(), 3))

Bei offenen Aufträgen fehlt das Lieferdatum zu Recht, bei fünf gelieferten ist die Lücke ein Fehler. Der Ausschuss `-1` heißt „nicht erfasst“ und muss vor jeder Rechnung zu einem fehlenden Wert werden, etwa mit `NULLIF(ausschuss_stueck, -1)`.

## 5 · Ausreißer und Plausibilität: IQR, MAD und Fachregeln
*Lerneinheit 5 · Skript Abschnitt 5*

- **IQR-Regel** nach Tukey (1977): auffällig sind Werte außerhalb von $[Q_1 - 1{,}5\,\mathrm{IQR},\; Q_3 + 1{,}5\,\mathrm{IQR}]$ mit $\mathrm{IQR} = Q_3 - Q_1$. Quartile liefert `serie.quantile(0.25)` und `serie.quantile(0.75)`.
- **Robuster z-Wert** nach Iglewicz und Hoaglin (1993): $M_i = 0{,}6745\,(x_i - \tilde x)/\mathrm{MAD}$ mit dem Median $\tilde x$ und $\mathrm{MAD} = \mathrm{median}_i\,|x_i - \tilde x|$. Auffällig ist $|M_i| > 3{,}5$.
- Der klassische z-Wert $(x_i - \bar x)/s$ wird vom Ausreißer selbst verdorben (**Maskierung**).
- Statistik meldet **Kandidaten**. Ob ein Kandidat ein Fehler ist, entscheiden **Fachregeln**: gefertigte Stück höchstens Kapazität, Ausschuss zwischen 0 und gefertigten Stück, Hallentemperatur zwischen 10 und 40 °C, Stillstand ein Vielfaches von 5 Minuten.

In [ ]:
# Die acht Schichten der Linie 2 aus dem Skript, Abschnitte 5.1 und 5.2
ist_acht = pd.Series([194, 182, 1980, 206, 120, 200, 212, 188])
q1, q3 = ist_acht.quantile(0.25), ist_acht.quantile(0.75)
iqr = q3 - q1
print("Quartile", q1, q3, "| Grenzen", q1 - 1.5 * iqr, q3 + 1.5 * iqr)
median = ist_acht.median()
mad = (ist_acht - median).abs().median()
m = 0.6745 * (ist_acht - median) / mad
z = (ist_acht - ist_acht.mean()) / ist_acht.std()
print(pd.DataFrame({"ist": ist_acht, "z": z.round(2), "m": m.round(2)}))

Die IQR-Regel und der robuste z-Wert melden 1980 und 120, der klassische z-Wert keinen. 1980 ist unmöglich, die Linie schafft höchstens 260 Stück. 120 ist ungewöhnlich, aber möglich, etwa nach einem langen Stillstand.

An der Rohdatenbank prüfst du die gefertigten Stück als Anteil an der Kapazität der Linie, damit alle vier Linien vergleichbar sind:

In [ ]:
schichten_k = sql("""
SELECT DISTINCT s.*, l.kapazitaet_stueck
FROM schichten AS s JOIN linien AS l ON s.linie = l.linie
""")
anteil = schichten_k["ist_stueck"] / schichten_k["kapazitaet_stueck"]
print(len(anteil), "Schichten, Anteil an der Kapazität von", round(anteil.min(), 3), "bis", round(anteil.max(), 3))

### TODO 8 · IQR-Regel, robuster z-Wert und Fachregel
1. `iqr_kandidat`: boolesche Maske der Schichten, deren `anteil` außerhalb der IQR-Grenzen liegt.
2. `mad_kandidat`: Maske der Schichten mit $|M_i| > 3{,}5$ für `anteil`.
3. `regel_verstoss`: Maske der Schichten, die mehr gefertigt haben, als die Linie kann.
4. `platzhalter`: Maske der Schichten in `schichten_k` mit negativem Ausschuss. `platzhalter_iqr`: wie viele davon die IQR-Regel angewandt auf `ausschuss_stueck` melden würde (eine ganze Zahl).

In [ ]:
# TODO 8
iqr_kandidat = None
mad_kandidat = None
regel_verstoss = None
platzhalter = None
platzhalter_iqr = None

if iqr_kandidat is not None:
    print(iqr_kandidat.sum(), mad_kandidat.sum(), regel_verstoss.sum())
print(platzhalter_iqr)

In [ ]:
check("TODO 8a iqr_kandidat", lambda: iqr_kandidat.sum() == 73)
check("TODO 8b mad_kandidat", lambda: mad_kandidat.sum() == 27)
check("TODO 8c regel_verstoss", lambda: regel_verstoss.sum() == 6 and (iqr_kandidat & regel_verstoss).sum() == 6
      and (mad_kandidat & regel_verstoss).sum() == 6)
check("TODO 8d Platzhalter", lambda: platzhalter.sum() == 5 and platzhalter_iqr == 0)

In [ ]:
if regel_verstoss is None:
    print("⏳ Erst TODO 8 lösen.")
else:
    def z_kandidaten(x):
        return int((((x - x.mean()) / x.std()).abs() > 3).sum())
    print("Klassischer z-Wert, alle Schichten:            ", z_kandidaten(anteil))
    print("Klassischer z-Wert ohne die sechs Regelverstöße:", z_kandidaten(anteil[~regel_verstoss]))
    print(schichten_k.loc[regel_verstoss, ["datum", "linie", "schicht", "soll_stueck", "ist_stueck", "kapazitaet_stueck"]])

Mit den sechs unmöglichen Werten meldet der klassische z-Wert nur diese sechs, ohne sie 33 auffällig schwache Schichten: Maskierung. Die sechs Werte haben eine Null zu viel, geteilt durch 10 liegen sie bei 72 % bis 95 % des Solls ihrer Schicht. Den Platzhalter `-1` findet nur die Fachregel, für die IQR-Regel liegt er mitten im Normalbereich.

Stillstandsdauern sind stark rechtsschief. Auf der ursprünglichen Skala hält die IQR-Regel jede längere Störung für auffällig, auf der Log-Skala nicht:

In [ ]:
dauer = stoerungen_roh["stillstand_min"]
for titel, x in [("Minuten", dauer), ("log(Minuten)", np.log(dauer))]:
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    unten, oben = x < q1 - 1.5 * (q3 - q1), x > q3 + 1.5 * (q3 - q1)
    print(f"{titel:<13} IQR-Regel: {unten.sum():>4} zu klein, {oben.sum():>4} zu groß")

Auf der Log-Skala bleiben 69 sehr lange Störungen und 239 Meldungen unter 3 Minuten, obwohl Störungen erst ab 5 Minuten erfasst werden. Der Median je Monat und Linie zeigt, woher sie kommen. Für das Muster reicht der Join über die richtig geschriebenen Kennungen:

In [ ]:
st_linie = sql("""
SELECT substr(s.datum, 1, 7) AS monat, m.linie, s.stillstand_min
FROM stoerungen AS s JOIN maschinen AS m ON s.maschine = m.maschine
""")
median_monat = st_linie.pivot_table(index="monat", columns="linie", values="stillstand_min", aggfunc="median")
print(median_monat.loc["2024-11":"2025-04"])
print(sql("""
SELECT linie, MIN(datum) AS von, MAX(datum) AS bis, COUNT(*) AS zeilen,
       MIN(hallentemp_c) AS min_temp, MAX(hallentemp_c) AS max_temp
FROM schichten
WHERE hallentemp_c > 40
GROUP BY linie
"""))

Zwei Einheitenfehler, beide auf Linie 3: Im Januar und Februar 2025 hat ein altes Erfassungsprogramm den Stillstand in Stunden gespeichert, und im Juni 2025 hat der Temperaturlogger der Halle 2 drei Wochen lang Grad Fahrenheit geschrieben.

### TODO 9 · Einheitenfehler finden und umrechnen
Die Zelle legt zuerst `temp` an, die Spalte `hallentemp_c` von `schichten_roh`. `dauer` ist der Stillstand aus der Zelle zur Log-Skala.
1. `fahrenheit`: Maske der Zeilen in `schichten_roh` mit einer Hallentemperatur über 40 °C.
2. `temp_korrigiert`: eine Kopie von `temp`, in der diese Werte mit $C = (F - 32) \cdot 5/9$ umgerechnet und auf eine Nachkommastelle gerundet sind. `serie.where(bedingung, anderer_wert)` behält die Werte, wo die Bedingung wahr ist, und setzt sonst den anderen Wert ein.
3. `in_stunden`: Maske der Meldungen in `stoerungen_roh`, deren Stillstand unter 5 Minuten liegt oder kein Vielfaches von 5 ist.
4. `stillstand_korrigiert`: der Stillstand, in dem diese Werte mal 60 genommen und auf ein Vielfaches von 5 gerundet sind: `(x * 60 / 5).round() * 5`.

In [ ]:
schichten_roh = sql("SELECT * FROM schichten")
temp = schichten_roh["hallentemp_c"]

# TODO 9
fahrenheit = None
temp_korrigiert = None
in_stunden = None
stillstand_korrigiert = None

print(temp_korrigiert)

In [ ]:
check("TODO 9a fahrenheit", lambda: fahrenheit.sum() == 39 and set(schichten_roh.loc[fahrenheit, "linie"]) == {3})
check("TODO 9b temp_korrigiert", lambda: temp_korrigiert.dropna().between(10, 40).all()
      and math.isclose(temp_korrigiert[fahrenheit].max(), 26.2)
      and temp_korrigiert.isna().sum() == schichten_roh["hallentemp_c"].isna().sum())
check("TODO 9c in_stunden", lambda: in_stunden.sum() == 241)
check("TODO 9d stillstand_korrigiert", lambda: (stillstand_korrigiert % 5 == 0).all() and stillstand_korrigiert.min() == 5
      and stillstand_korrigiert.sum() == 551715)

Vorsicht mit `%` in SQL: SQLite schneidet beide Zahlen vorher auf ganze Zahlen ab, `0.75 % 5` ergibt dort 0, `2.0 % 5` dagegen 2. Die Regel `stillstand_min % 5 != 0` findet in SQL deshalb nur die 45 Stundenwerte ab einer Stunde, die 196 unter einer Stunde fehlen. pandas rechnet `%` mit Gleitkommazahlen.

In [ ]:
print(sql("SELECT SUM(stillstand_min % 5 != 0) AS nur_rest, SUM(stillstand_min < 5 OR stillstand_min % 5 != 0) AS mit_untergrenze FROM stoerungen"))

## 6 · Datenprüfungen als Tests und das Datenblatt
*Lerneinheit 6 · Skript Abschnitt 6*

Damit Prüfungen bei jedem Abzug wieder laufen, schreibst du sie als Code. Ein Dictionary ordnet jeder Regel, in Worten beschrieben, eine Funktion zu, die zu einer Tabelle eine boolesche Maske der **Verstöße** liefert. `lambda t: …` ist eine Funktion ohne Namen mit der Tabelle `t` als Argument. Fehlende Werte sind dabei kein Verstoß, sie prüfst du getrennt.

In [ ]:
probe = pd.DataFrame({
    "datum": ["2025-06-02", "2025-06-02", "03.06.2025", "2025-06-03", "2025-06-03", "2025-06-04"],
    "linie": [3, 1, 2, 3, 4, 1],
    "schicht": ["Früh", "Spät", "Nacht", "spät", "Früh", "Früh"],
    "ist_stueck": [210, 3300, 160, 205, 140, 350],
    "ausschuss_stueck": [4, 9, -1, 3, 2, 8],
    "hallentemp_c": [71.6, 22.4, 21.9, 22.1, None, 23.0],
})
linien = sql("SELECT * FROM linien")
KAPAZITAET = linien.set_index("linie")["kapazitaet_stueck"]

REGELN_SCHICHTEN = {
    "Schlüssel eindeutig": lambda t: t.duplicated(["datum", "linie", "schicht"]),
    "datum als JJJJ-MM-TT": lambda t: ~t["datum"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}"),
    "schicht Früh, Spät oder Nacht": lambda t: ~t["schicht"].isin(["Früh", "Spät", "Nacht"]),
    "ist höchstens Kapazität": lambda t: t["ist_stueck"] > t["linie"].map(KAPAZITAET),
    "Ausschuss von 0 bis ist": lambda t: (t["ausschuss_stueck"] < 0) | (t["ausschuss_stueck"] > t["ist_stueck"]),
    "Temperatur 10 bis 40 °C": lambda t: (t["hallentemp_c"] < 10) | (t["hallentemp_c"] > 40),
}

for regel, test in REGELN_SCHICHTEN.items():
    print(f"{regel:<31} Verstöße in Zeile {probe.index[test(probe)].tolist()}")

Im März 2025 steht das Datum als `TT.MM.JJJJ` in der Datenbank. Mit `format="mixed"` rät pandas das Format für jeden Wert einzeln, und das geht still schief:

In [ ]:
probe_datum = pd.Series(["2025-02-28", "03.03.2025", "05.03.2025", "13.03.2025", "2025-04-01"])
print(pd.to_datetime(probe_datum, format="mixed").dt.strftime("%Y-%m-%d").tolist())
print(pd.to_datetime(probe_datum, format="mixed", dayfirst=True).dt.strftime("%Y-%m-%d").tolist())

Ohne `dayfirst` wird aus dem 5. März der 3. Mai. Mit `dayfirst=True` macht pandas 3.0 in Colab aus dem 1. April den 4. Januar, pandas 2.3 im Browser liest ihn richtig. Raten ist also keine Strategie.

### TODO 10 · Datum und Schicht vereinheitlichen
Das Skript zeigt die Methode in Abschnitt 6.2 an anderen Daten, hier überträgst du sie auf die Rohdatenbank.
1. `vereinheitliche_datum(spalte)`: liest eine Series von Datumstexten in den Formaten `JJJJ-MM-TT` und `TT.MM.JJJJ`, jedes mit festem `format` und `errors="coerce"`, und führt beide Ergebnisse mit `fillna` zusammen.
2. `SCHICHT_NAMEN`: ein Dictionary, das jede klein geschriebene Schreibweise (`"früh"`, `"frueh"`, `"spät"`, `"spaet"`, `"nacht"`) auf `"Früh"`, `"Spät"` oder `"Nacht"` abbildet.
3. `vereinheitliche_schicht(spalte)`: entfernt Leerzeichen am Rand, schreibt klein, übersetzt mit `.map(SCHICHT_NAMEN)` und wandelt in `SCHICHT_TYP` um.

In [ ]:
# TODO 10
def vereinheitliche_datum(spalte):
    return spalte          # ersetze diese Zeile

SCHICHT_NAMEN = {}

def vereinheitliche_schicht(spalte):
    return spalte          # ersetze diese Zeile

print(vereinheitliche_datum(probe_datum))

In [ ]:
check("TODO 10a vereinheitliche_datum", lambda: vereinheitliche_datum(probe_datum).dt.strftime("%Y-%m-%d").tolist()
      == ["2025-02-28", "2025-03-03", "2025-03-05", "2025-03-13", "2025-04-01"])
check("TODO 10b Datum der Rohdaten", lambda: vereinheitliche_datum(schichten_roh["datum"]).notna().all()
      and vereinheitliche_datum(schichten_roh["datum"]).between("2025-03-01", "2025-03-31").sum() == 268)
check("TODO 10c vereinheitliche_schicht", lambda: vereinheitliche_schicht(pd.Series([" Früh", "Frueh", "spät", "Spaet", "Nacht "])).tolist()
      == ["Früh", "Früh", "Spät", "Spät", "Nacht"])
check("TODO 10d Schicht der Rohdaten", lambda: vereinheitliche_schicht(schichten_roh["schicht"]).value_counts(sort=False).to_dict()
      == {"Früh": 2032, "Spät": 1892, "Nacht": 1892} and isinstance(vereinheitliche_schicht(schichten_roh["schicht"]).dtype, pd.CategoricalDtype))

Bleiben die 195 inhaltsgleichen Störungen aus TODO 6. Sie einfach zu löschen, scheint naheliegend. In pandas markiert `duplicated` wie `ROW_NUMBER() > 1` jede Zeile, die mit kleinerer Kennung schon einmal vorkam. Der Abgleich mit Modul 2 zeigt, was das Löschen anrichtet, und die Kennungen verraten die Kopien:

In [ ]:
inhalt = [s for s in stoerungen_roh.columns if s != "id"]
nach_id = stoerungen_roh.sort_values("id")
doppelt = nach_id.duplicated(inhalt)
entfernt = nach_id.loc[doppelt, "id"]
print("Nach dem Löschen aller Dubletten:", len(stoerungen_roh) - doppelt.sum(), "Meldungen, in Modul 2:", len(stoerungen_m2))
print("Gelöschte Kennungen, die es in Modul 2 gibt:", entfernt.isin(stoerungen_m2["id"]).sum())

datum = pd.to_datetime(nach_id["datum"], format="%Y-%m-%d")
nachimport = datum < datum.cummax().shift()     # Datum liegt vor dem einer kleineren Kennung
print("Meldungen außerhalb der zeitlichen Reihenfolge:", nachimport.sum(), nach_id.loc[nachimport, "id"].min(), "bis", nach_id.loc[nachimport, "id"].max())
print("davon inhaltsgleich mit einer früheren Meldung:", (nachimport & doppelt).sum())

24 der 195 Dubletten sind echte Meldungen: zwei gleiche Störungen derselben Maschine in derselben Schicht. Die übrigen 171 stammen aus einem Block am Ende der Tabelle, dessen Datum wieder im Oktober 2024 beginnt, offenbar ein Nachimport. Ohne eindeutigen Schlüssel ist jede Dublettenregel eine Annahme. Diese hier lautet: Kopie ist, was inhaltsgleich ist und aus dem Nachimport stammt.

Die nächste Zelle fasst alle Schritte in zwei Funktionen zusammen, mit deinen Funktionen aus TODO 4 und TODO 10. `assert bedingung` bricht mit einem Fehler ab, wenn die Bedingung falsch ist. So endet ein späterer Abzug mit neuen Fehlerarten laut, statt still falsche Zahlen zu liefern. Die Funktionen ändern die Rohdaten nicht, und fehlende Werte bleiben fehlend.

In [ ]:
def bereinige_schichten(roh, linien):
    """Bereinigt einen Abzug der Tabelle schichten (Skript, Abschnitte 3 bis 6)."""
    t = roh.drop_duplicates().reset_index(drop=True)                  # exakte Kopien (3.4)
    t["datum"] = vereinheitliche_datum(t["datum"])                     # zwei Datumsformate (6.2)
    t["schicht"] = vereinheitliche_schicht(t["schicht"])               # elf Schreibweisen (6.2)
    f = t["hallentemp_c"] > 40                                         # Grad Fahrenheit (5.5)
    t.loc[f, "hallentemp_c"] = ((t.loc[f, "hallentemp_c"] - 32) * 5 / 9).round(1)
    t["ausschuss_stueck"] = t["ausschuss_stueck"].where(t["ausschuss_stueck"] >= 0)   # Platzhalter (4.4)
    kapazitaet = t["linie"].map(linien.set_index("linie")["kapazitaet_stueck"])
    zu_gross = t["ist_stueck"] > kapazitaet                            # eine Null zu viel (5.4)
    t.loc[zu_gross, "ist_stueck"] = t.loc[zu_gross, "ist_stueck"] // 10
    assert t["datum"].notna().all() and t["schicht"].notna().all()
    assert not t.duplicated(["datum", "linie", "schicht"]).any()
    assert (t["ist_stueck"] <= kapazitaet).all() and t["hallentemp_c"].dropna().between(10, 40).all()
    return t


def bereinige_stoerungen(roh, maschinen):
    """Bereinigt einen Abzug der Tabelle stoerungen (Skript, Abschnitte 3 bis 6)."""
    t = roh.copy()
    t["maschine"] = normiere_maschine(t["maschine"])                   # Schreibweisen (3.2)
    t["datum"] = pd.to_datetime(t["datum"], format="%Y-%m-%d")
    t["schicht"] = t["schicht"].astype(SCHICHT_TYP)
    t = t.sort_values("id")
    doppelt = t.duplicated([s for s in t.columns if s != "id"])
    nachimport = t["datum"] < t["datum"].cummax().shift()
    t = t[~(doppelt & nachimport)].copy()                              # Kopien aus dem Nachimport (6.3)
    h = (t["stillstand_min"] < 5) | (t["stillstand_min"] % 5 != 0)     # Stunden statt Minuten (5.5)
    t.loc[h, "stillstand_min"] = (t.loc[h, "stillstand_min"] * 60 / 5).round() * 5
    t["stillstand_min"] = t["stillstand_min"].astype("int64")
    t["behoben"] = t["behoben"].astype(bool)
    t = t.reset_index(drop=True)
    assert t["id"].is_unique and t["maschine"].isin(maschinen["maschine"]).all()
    assert (t["stillstand_min"] % 5 == 0).all() and t["stillstand_min"].between(5, 480).all()
    return t


def bausteine_fertig():
    """Prüft, ob die Funktionen aus TODO 4 und TODO 10 das Erwartete liefern."""
    try:
        return (normiere_maschine(pd.Series(["M3", "m-12"])).tolist() == ["M-03", "M-12"]
                and vereinheitliche_datum(pd.Series(["05.03.2025"])).iloc[0] == pd.Timestamp("2025-03-05")
                and vereinheitliche_schicht(pd.Series([" Spaet"])).iloc[0] == "Spät")
    except Exception:
        return False


if bausteine_fertig():
    schichten_sauber = bereinige_schichten(schichten_roh, linien)
    stoerungen_sauber = bereinige_stoerungen(stoerungen_roh, maschinen)
    print("schichten_sauber:", schichten_sauber.shape, "| stoerungen_sauber:", stoerungen_sauber.shape)
else:
    schichten_sauber = stoerungen_sauber = None
    print("⏳ Erst TODO 4 und TODO 10 lösen.")

### TODO 11 · Prüfbericht und Abgleich mit Modul 2
1. `pruefbericht(name, tabelle, regeln)`: eine Funktion, die jede Regel aus `regeln` auf `tabelle` anwendet und einen DataFrame mit einer Zeile je Regel liefert, mit den Spalten `regel`, `tabelle` (der übergebene Name), `verstoesse` (Anzahl) und `beispiel` (zum Beispiel `"Zeile 17"` für die erste betroffene Zeile, sonst `""`). `maske.idxmax()` liefert die Beschriftung des ersten `True`.
2. `bericht`: die Berichte für `schichten_roh` (Name `"schichten"`) und für `schichten_sauber` (Name `"schichten_sauber"`) untereinander, mit `pd.concat([...], ignore_index=True)`.
3. `abweichung`: Verknüpfe `schichten_sauber` mit `produktion` über `datum`, `linie` und `schicht` (`validate="one_to_one"`, `suffixes=("", "_m2")`) und zähle für die Spalten `soll_stueck`, `ist_stueck`, `ausschuss_stueck`, `personal`, `auslastung` und `hallentemp_c`, in wie vielen Zeilen der Wert von dem in Modul 2 abweicht (`!=`). Speichere das als Series mit dem Spaltennamen als Index.

In [ ]:
# TODO 11
def pruefbericht(name, tabelle, regeln):
    return None            # ersetze diese Zeile

bericht = None
abweichung = None

print(bericht)
print(abweichung)

In [ ]:
check("TODO 11a pruefbericht", lambda: list(pruefbericht("probe", probe, REGELN_SCHICHTEN).columns) == ["regel", "tabelle", "verstoesse", "beispiel"]
      and pruefbericht("probe", probe, REGELN_SCHICHTEN)["verstoesse"].tolist() == [0, 1, 1, 1, 1, 1]
      and pruefbericht("probe", probe, REGELN_SCHICHTEN)["beispiel"].tolist()[:2] == ["", "Zeile 2"])
check("TODO 11b bericht", lambda: len(bericht) == 12
      and bericht.groupby("tabelle")["verstoesse"].sum().to_dict() == {"schichten": 466, "schichten_sauber": 0})
check("TODO 11c abweichung", lambda: abweichung.to_dict() == {"soll_stueck": 0, "ist_stueck": 0, "ausschuss_stueck": 5,
      "personal": 74, "auslastung": 0, "hallentemp_c": 302})

In [ ]:
if stoerungen_sauber is None:
    print("⏳ Erst TODO 4 und TODO 10 lösen.")
else:
    vergleich_st = stoerungen_sauber.merge(stoerungen_m2, on="id", how="outer", suffixes=("", "_m2"),
                                           indicator=True, validate="one_to_one")
    print(vergleich_st["_merge"].value_counts().to_dict())
    print({s: int((vergleich_st[s] != vergleich_st[s + "_m2"]).sum())
           for s in ["datum", "schicht", "maschine", "kategorie", "stillstand_min", "beschreibung", "behoben"]})
    maschinen_m2 = pd.read_csv("daten/maschinen.csv")
    print((maschinen != maschinen_m2).sum().to_dict())

Die Bereinigung trifft in beiden Tabellen alles, was sich beheben lässt: Bei den Schichten weichen nur die 5 Platzhalter, 74 fehlende Personalzahlen und 302 fehlende Temperaturen ab, bei den Störungen nur 1.502 fehlende Kategorien. Diese Werte sind verloren. In den Stammdaten endet der Name von M-12 mit einem Leerzeichen, und das Baujahr von M-20 fehlt.

Was nicht zu beheben ist, gehört ins **Datenblatt** (Gebru et al. 2021). Es beantwortet Fragen in sieben Gruppen: Motivation, Zusammensetzung, Erhebung, Vorverarbeitung, Verwendung, Weitergabe und Pflege. Für `schichten_sauber` gehören mindestens hinein: Herkunft und Abzugsdatum, jeder Bereinigungsschritt mit Anzahl, die Lücken mit ihrem vermuteten Mechanismus und die Folge für Auswertungen, etwa dass Sommermittel der Temperatur zu niedrig sind.

## 7 · Daten über Menschen: Pseudonymisierung, Datenminimierung, Grundsätze der DSGVO
*Lerneinheit 7 · Skript Abschnitt 7*

- **Personenbezogene Daten** (Art. 4 Nr. 1 DSGVO) sind alle Informationen über eine identifizierte oder identifizierbare Person. **Direkte Kennungen** wie Name, E-Mail oder Personalnummer bestimmen eine Person allein, **Quasi-Kennungen** wie Geburtsdatum, Linie, Schichtgruppe oder Eintritt in Kombination.
- **Grundsätze** (Art. 5): Rechtmäßigkeit, Zweckbindung, Datenminimierung, Richtigkeit, Speicherbegrenzung, Integrität und Vertraulichkeit, dazu die Rechenschaftspflicht.
- **Pseudonymisierung** (Art. 4 Nr. 5) ersetzt Kennungen so, dass eine Zuordnung nur mit getrennt aufbewahrter Zusatzinformation möglich ist. Pseudonymisierte Daten bleiben personenbezogen, erst **anonyme** Daten fallen nicht unter die DSGVO (Erwägungsgrund 26).
- Ein **gesalzener Hash** rechnet das Pseudonym aus der Kennung und einem geheimen Salz. Ohne Salz kann jeder die Hashes aller denkbaren Kennungen ausrechnen.
- **k-Anonymität** (Sweeney 2002): Jede Kombination der Quasi-Kennungen kommt mindestens $k$-mal vor. $k$ ist die kleinste Gruppengröße.

Das ist keine Rechtsberatung. Bevor du im Verein oder im Betrieb mit Personendaten arbeitest, fragst du die oder den Datenschutzbeauftragten. Die Personen in der Datenbank sind erfunden, du behandelst sie trotzdem wie echte Beschäftigte.

In [ ]:
import hashlib

# Die sechs erfundenen Beschäftigten aus dem Skript, Abschnitte 7.3 bis 7.5
personen = pd.DataFrame({
    "personal_nr": ["P-1", "P-2", "P-3", "P-4", "P-5", "P-6"],
    "name": ["Anna Brenner", "Ben Conrad", "Clara Dietz", "David Eberle", "Elif Fink", "Felix Gerber"],
    "geburtsdatum": ["1990-05-14", "1987-11-30", "1968-02-17", "1971-09-21", "1999-01-08", "2001-06-04"],
    "linie": [1, 1, 2, 2, 2, 2],
    "schichtgruppe": ["A", "B", "A", "A", "B", "A"],
})
SALZ = "nur-ein-beispiel"   # in echt: secrets.token_hex(16), getrennt von den Daten aufbewahren

def pseudonym(kennung, salz=SALZ):
    """Gesalzener SHA-256-Hash einer Kennung, gekürzt auf 12 Zeichen."""
    return hashlib.sha256((salz + kennung).encode("utf-8")).hexdigest()[:12]

personen["pseudonym"] = personen["personal_nr"].map(pseudonym)
personen["alter"] = 2026 - personen["geburtsdatum"].str[:4].astype("int64")
GRUPPEN = [0, 30, 40, 50, 60, 100]
NAMEN = ["unter 30", "30 bis 39", "40 bis 49", "50 bis 59", "ab 60"]
personen["altersgruppe"] = pd.cut(personen["alter"], bins=GRUPPEN, right=False, labels=NAMEN)
minimal = personen[["pseudonym", "altersgruppe", "linie", "schichtgruppe"]]
print(minimal)
print("k mit Schichtgruppe:", minimal.groupby(["altersgruppe", "linie", "schichtgruppe"], observed=True).size().min())
print("k ohne Schichtgruppe:", minimal.groupby(["altersgruppe", "linie"], observed=True).size().min())

### TODO 12 · Pseudonymisieren, verallgemeinern, k-Anonymität prüfen
Die Werksleitung möchte die Altersstruktur und die Qualifikationen je Linie kennen. Arbeite mit `bedienende` aus der Rohdatenbank:
1. Ergänze in `bedienende` die Spalten `alter` (2026 minus Geburtsjahr) und `altersgruppe` (mit `pd.cut`, `GRUPPEN` und `NAMEN` wie oben).
2. `bedienende_min`: eine Tabelle mit genau den Spalten `pseudonym` (gesalzener Hash der `personal_nr` mit der Funktion `pseudonym`), `altersgruppe`, `linie`, `schichtgruppe` und `qualifikation`. Namen, Kontaktdaten, Geburtsdatum und Eintritt bleiben draußen.
3. `k_fein`: das $k$ von `bedienende_min` für die Quasi-Kennungen `altersgruppe`, `linie` und `schichtgruppe`. `k_grob`: das $k$ für `altersgruppe` und `linie`.

In [ ]:
bedienende = sql("SELECT * FROM bedienende")

# TODO 12
bedienende_min = None
k_fein = None
k_grob = None

print(bedienende_min)
print(k_fein, k_grob)

In [ ]:
check("TODO 12a Altersgruppen", lambda: bedienende["altersgruppe"].value_counts(sort=False).tolist() == [30, 35, 22, 26, 13])
check("TODO 12b bedienende_min", lambda: bedienende_min.shape == (126, 5)
      and sorted(bedienende_min.columns) == ["altersgruppe", "linie", "pseudonym", "qualifikation", "schichtgruppe"]
      and bedienende_min["pseudonym"].is_unique and bedienende_min["pseudonym"].iloc[0] == pseudonym("P-1001"))
check("TODO 12c k_fein und k_grob", lambda: k_fein == 1 and k_grob == 2)

In [ ]:
if bedienende_min is None:
    print("⏳ Erst TODO 12 lösen.")
else:
    gruppen_fein = bedienende_min.groupby(["altersgruppe", "linie", "schichtgruppe"], observed=True).size()
    print("Personen allein in ihrer Gruppe (Altersgruppe, Linie, Schichtgruppe):", (gruppen_fein == 1).sum())
    print(bedienende_min.groupby(["altersgruppe", "linie"], observed=True).size().unstack())

Mit der Schichtgruppe sind 13 Personen allein in ihrer Gruppe. Wer Altersgruppe, Linie und Schichtgruppe einer dieser Personen kennt, findet ihre Zeile und damit ihre Qualifikation. Ohne Schichtgruppe ist $k = 2$. Welches $k$ genügt, hängt von Zweck und Risiko ab, oft wird $k \geq 5$ verlangt. Dafür müsstest du weiter verallgemeinern, etwa breitere Altersgruppen bilden.

## Bonus

### Bonus B1 · Termintreue der Aufträge
Wie viele gelieferte Aufträge kamen spätestens zum zugesagten `liefertermin`? Lass dabei die Aufträge weg, bei denen das Lieferdatum fehlt oder der Liefertermin vor dem Bestelldatum liegt (dort sind offenbar zwei Werte vertauscht). Datumstexte im Format `JJJJ-MM-TT` darfst du direkt vergleichen.
1. `termintreue`: der Anteil pünktlicher Aufträge als Zahl zwischen 0 und 1, mit einer SQL-Abfrage.
2. `je_monat`: derselbe Anteil je Bestellmonat (`substr(bestellt_am, 6, 2)`) als DataFrame mit den Spalten `monat` und `quote`.

In [ ]:
# Bonus B1
termintreue = None
je_monat = None

print(termintreue)
print(je_monat)

In [ ]:
check("B1 termintreue", lambda: math.isclose(termintreue, 0.697623, abs_tol=1e-6))
check("B1 je_monat", lambda: len(je_monat) == 12 and math.isclose(je_monat.set_index("monat").loc["01", "quote"], 0.475728, abs_tol=1e-6))

Gut zwei Drittel der Aufträge kommen pünktlich. Im November, Dezember und Januar bestellte Aufträge kommen nur etwa zur Hälfte rechtzeitig. Ob das an Feiertagen, Betriebsruhe oder Auslastung liegt, zeigt diese Tabelle allein nicht.

### Bonus B2 · Ein Datenblatt für `schichten_sauber`
Schreibe ein Dictionary `datenblatt` mit den sieben Fragengruppen nach Gebru et al. (2021) als Schlüssel: `"Motivation"`, `"Zusammensetzung"`, `"Erhebung"`, `"Vorverarbeitung"`, `"Verwendung"`, `"Weitergabe"` und `"Pflege"`. Jeder Wert ist ein Text mit mindestens einem vollständigen Satz. Nenne bei der Vorverarbeitung jeden Bereinigungsschritt mit Anzahl und bei der Zusammensetzung die Lücken mit ihrem Mechanismus.

In [ ]:
# Bonus B2
datenblatt = {}

for frage, antwort in datenblatt.items():
    print(f"{frage}: {antwort}")

In [ ]:
FRAGEN = ["Motivation", "Zusammensetzung", "Erhebung", "Vorverarbeitung", "Verwendung", "Weitergabe", "Pflege"]
check("B2 sieben Fragengruppen beantwortet", lambda: sorted(datenblatt) == sorted(FRAGEN)
      and all(isinstance(datenblatt[f], str) and len(datenblatt[f].split()) >= 5 for f in FRAGEN))

## Zum Nachdenken

1. In TODO 2 fehlten 156 Meldungen, ohne dass eine Abfrage einen Fehler meldete. Welche zwei Schritte in einer Auswertung hätten dich darauf gestoßen?
2. Die Kategorie fehlt nachts häufiger. Unter welcher Bedingung wäre das MNAR statt MAR, und könntest du das an den Daten erkennen?
3. Der robuste z-Wert meldet 27 Schichten, die Fachregel nur 6. Was machst du mit den übrigen 21?
4. Deine Dublettenregel für die Störungen beruht auf einer Annahme. Woran würdest du merken, dass ein künftiger Abzug sie verletzt?
5. `bedienende_min` enthält kein Geburtsdatum und keinen Namen. Ist die Tabelle damit anonym im Sinne der DSGVO?

<details><summary>Hinweise zu den Antworten</summary>

1. Die Zeilenzahl vor und nach jedem Join oder Filter vergleichen, und ein Anti-Join, der Schlüssel ohne Partner zeigt. `HAVING` hat die abweichenden Kennungen zusätzlich versteckt.
2. Wenn bestimmte Kategorien öfter leer bleiben als andere, etwa weil Bedienfehler ungern eingetragen werden, hängt die Lücke vom fehlenden Wert selbst ab. Aus den Rohdaten allein lässt sich das nicht erkennen, dafür brauchst du zusätzliche Information, etwa eine Stichprobe nachträglich geklärter Meldungen.
3. Prüfen, nicht löschen. Es sind sehr schwache Schichten, möglich nach langem Stillstand. Ein Blick auf die Störungen derselben Schicht klärt, ob es eine Erklärung gibt.
4. Inhaltsgleiche Meldungen innerhalb der zeitlichen Reihenfolge, die nicht direkt aufeinanderfolgen, oder ein Nachimport ohne Dubletten wären Warnsignale. Eine Prüfregel im Bericht kann beides zählen.
5. Nein. Die Pseudonyme lassen sich mit dem Salz zurückführen, und 13 Personen sind über Altersgruppe, Linie und Schichtgruppe allein in ihrer Gruppe. Die Tabelle bleibt personenbezogen.
</details>

## Wie es weitergeht

Deine bereinigten Tabellen stimmen jetzt mit den geprüften aus Modul 2 überein, bis auf Lücken, die du kennst und benennen kannst. **Modul 4 „Explorative Analyse und Visualisierung“** fragt, was in den Daten steckt, bevor du ein Modell baust: Verteilungen, Zusammenhänge und Grafiken, die Muster sichtbar machen. Vieles aus diesem Modul hilft dir dort, etwa die Log-Skala für schiefe Größen und der kritische Blick auf jede Zahl, die zu glatt aussieht.